# nnU-Net: Nodule Segmentation on RexGrounding-CT

This notebook covers:
1. Environment setup and variable configuration
2. Dataset conversion from RexGrounding-CT format → nnU-Net format
3. Dataset fingerprinting and preprocessing
4. Model training (all folds)
5. Finding the best configuration
6. Inference and postprocessing

---
**Dataset specifics handled here:**
- Multi-finding masks (FxHxWxD) — only GGO channels extracted
- Instance-coded mask values → binary semantic mask (0/1)
- Variable depth (D: 104–1005 slices)
- HU-valued CT volumes
- JSON-driven train/valid/test splits

## 0. Install dependencies

In [ ]:
# Install PyTorch first (edit the index-url to match your CUDA version)
# Check https://pytorch.org/get-started/locally/ for the right command
# Example for CUDA 12.1:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Then install nnU-Net
!pip install nnunetv2

# Optional: network topology visualisation
# !pip install --upgrade git+https://github.com/FabianIsensee/hiddenlayer.git

## 1. Configuration — edit these paths before running anything else

In [ ]:
import os
from pathlib import Path

# Path to the JSON split file produced for Nodules
SPLIT_JSON = Path("/home/chest_ct/code/data/rexgrounding-ct/dataset_2d.json")

# Root directories where raw data already lives
CT_ROOT    = Path("/home/chest_ct/code/data/data_volumes/dataset/train_fixed") # parent of train_XXX folders
SEG_ROOT   = Path("/home/chest_ct/code/data/segmentations/segmentations")      # flat folder of masks

# nnU-Net storage locations
NNUNET_RAW          = Path("/home/chest_ct/code/models/nnu-net/storage/nnUNet_raw")
NNUNET_PREPROCESSED = Path("/home/chest_ct/code/models/nnu-net/storage/nnUNet_preprocessed")
NNUNET_RESULTS      = Path("/home/chest_ct/code/models/nnu-net/storage/nnUNet_results")

# Dataset identity
DATASET_ID   = 104          # any unused 3-digit integer
DATASET_NAME = "Nodules"        # short, no spaces
DATASET_FULL_NAME = f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
DATASET_RAW_DIR   = NNUNET_RAW / DATASET_FULL_NAME

images_tr = Path(DATASET_RAW_DIR) / "imagesTr"
labels_tr = Path(DATASET_RAW_DIR) / "labelsTr"
images_ts = Path(DATASET_RAW_DIR) / "imagesTs"
labels_ts = Path(DATASET_RAW_DIR) / "labelsTs"


for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    os.makedirs(d, exist_ok=True)

# Set environment variables for the current process and all child processes
os.environ["nnUNet_raw"]          = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"]      = str(NNUNET_RESULTS)

print("Dataset folder:", DATASET_RAW_DIR)
print("Environment variables set.")

Dataset folder: /home/chest_ct/code/models/nnu-net/storage/nnUNet_raw/Dataset104_Nodules
Environment variables set.


## 2. Convert RexGrounding-CT → nnU-Net format

In [ ]:
import shutil
import numpy as np
import nibabel as nib
from tqdm.auto import tqdm


def extract_nodule_mask(seg_path: str, nodule_channels: list[int]) -> np.ndarray:
    """
    Load a multi-finding mask (F x H x W x D) and collapse the Nodule
    channels into a single binary semantic mask (H x W x D).

    Background = 0, Nodule foreground = 1.
    Instance labels (>0) in any Nodule channel are all treated as foreground.
    """
    img  = nib.load(seg_path)
    data = img.get_fdata(dtype=np.float32)  # shape: F x H x W x D  (or H x W x D if F=1)

    # Union of all requested Nodule channels
    binary = np.zeros(data.shape[1:], dtype=np.uint8)  # H x W x D
    for ch in nodule_channels:
        if ch < data.shape[0]:
            binary = np.logical_or(binary, data[ch] > 0).astype(np.uint8)

    return binary, img.affine, img.header

In [ ]:
# ── Convert training + validation (both go into imagesTr / labelsTr) ──
# nnU-Net manages its own CV splits, so train+valid are merged here.
# Test images go into imagesTs (no labels required).

import json

images_tr = Path(DATASET_RAW_DIR) / "imagesTr"
labels_tr = Path(DATASET_RAW_DIR) / "labelsTr"
images_ts = Path(DATASET_RAW_DIR) / "imagesTs"
labels_ts = Path(DATASET_RAW_DIR) / "labelsTs"

split_data = json.load(open(SPLIT_JSON, "r"))
train_cases = split_data.get("train", [])
test_cases = split_data.get("test", [])

print(f"  Training cases : {len(train_cases)}")
print(f"  Test images    : {len(test_cases)}")

  Training cases : 443
  Test images    : 100


In [12]:
# ── Write dataset.json ─────────────────────────────────────────────
# nnU-Net v2 requires this file in the dataset root.

from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json

generate_dataset_json(
    output_folder   = DATASET_RAW_DIR,
    channel_names   = {0: "CT"},     # single CT modality; nnU-Net will use CT-specific normalisation
    labels          = {"background": 0, "GGO": 1},
    num_training_cases = len(train_cases),
    file_ending     = ".nii.gz",
    dataset_name    = DATASET_FULL_NAME,
    reference       = "RexGrounding-CT",
    description     = "Binary GGO segmentation extracted from multi-finding RexGrounding-CT masks",
)

print("dataset.json written to:", DATASET_RAW_DIR)

dataset.json written to: /home/chest_ct/code/models/nnu-net/storage/nnUNet_raw/Dataset104_Nodules


In [ ]:
for p in [images_tr, labels_tr, images_ts, labels_ts]:
    p.mkdir(parents=True, exist_ok=True)

# =========================
# Helpers
# =========================
def get_volume_path(case_name: str) -> Path:
    """
    Example:
    train_1696_a_2.nii.gz
    -> /train_fixed/train_1696/train_1696_a/train_1696_a_2.nii.gz
    """
    case_name = str(case_name)
    stem = case_name.replace(".nii.gz", "")   # train_1696_a_2
    parts = stem.split("_")                   # ['train', '1696', 'a', '2']

    patient_id = f"{parts[0]}_{parts[1]}"     # train_1696
    study_id   = f"{parts[0]}_{parts[1]}_{parts[2]}"  # train_1696_a

    return Path(CT_ROOT) / patient_id / study_id / case_name


def get_mask_path(case_name: str) -> Path:
    return Path(SEG_ROOT) / str(case_name)


def nnunet_image_name(case_name: str) -> str:
    """
    nnU-Net image names require _0000 before .nii.gz for single-channel CT.
    """
    return case_name.replace(".nii.gz", "_0000.nii.gz")


def copy_case(case_name: str, image_out_dir: Path, label_out_dir: Path):
    src_img = get_volume_path(case_name)
    src_lbl = get_mask_path(case_name)

    dst_img = image_out_dir / nnunet_image_name(case_name)
    dst_lbl = label_out_dir / case_name

    if not src_img.exists():
        print(f"[MISSING CT]   {src_img}")
        return False

    if not src_lbl.exists():
        print(f"[MISSING MASK] {src_lbl}")
        return False

    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_lbl, dst_lbl)

    return True


# =========================
# Load JSON
# =========================
with open(SPLIT_JSON, "r") as f:
    dataset = json.load(f)

# Expected:
# {
#   "train": ["train_1696_a_2.nii.gz", ...],
#   "test": [...]
# }

# =========================
# Copy files
# =========================
counts = {
    "train": 0,
    "val": 0,
    "test": 0,
    "missing": 0,
}

# nnU-Net usually puts train + val into imagesTr / labelsTr
for split in ["train", "val"]:
    for case_name in dataset.get(split, []):
        ok = copy_case(case_name, images_tr, labels_tr)
        if ok:
            counts[split] += 1
        else:
            counts["missing"] += 1

# Test goes into imagesTs / labelsTs
# Note: labelsTs is optional for real blind test inference,
# but useful if you want evaluation.
for case_name in dataset.get("test", []):
    ok = copy_case(case_name, images_ts, labels_ts)
    if ok:
        counts["test"] += 1
    else:
        counts["missing"] += 1

print("Copy completed.")
print(counts)

print("\nOutput folders:")
print("imagesTr:", len(list(images_tr.glob("*.nii.gz"))))
print("labelsTr:", len(list(labels_tr.glob("*.nii.gz"))))
print("imagesTs:", len(list(images_ts.glob("*.nii.gz"))))
print("labelsTs:", len(list(labels_ts.glob("*.nii.gz"))))

[MISSING MASK] /home/chest_ct/code/data/segmentations/train_1696_a_2.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_2381_a_2.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_1606_a_2.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_1860_a_1.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_1427_a_2.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_2111_a_1.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_2039_a_2.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_1580_a_2.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_1905_a_2.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_2314_a_1.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_2149_a_1.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_1514_a_2.nii.gz
[MISSING MASK] /home/chest_ct/code/data/segmentations/train_2160_a_2.nii.gz
[MISSING MAS

In [ ]:
# ── Sanity-check: inspect a single converted case ──────────────────
import nibabel as nib
import numpy as np

sample_case = train_cases[0]
ct_img  = nib.load(str(images_tr / nnunet_image_name(sample_case)))
lbl_img = nib.load(str(labels_tr / sample_case))

ct_arr  = ct_img.get_fdata()
lbl_arr = lbl_img.get_fdata()

print(f"Case : {sample_case}")
print(f"CT   shape : {ct_arr.shape}  | dtype: {ct_arr.dtype}")
print(f"Label shape: {lbl_arr.shape} | dtype: {lbl_arr.dtype}")
print(f"CT HU range: [{ct_arr.min():.0f}, {ct_arr.max():.0f}]")
print(f"Label unique values: {np.unique(lbl_arr)}")
print(f"GGO voxel fraction : {lbl_arr.mean():.4%}")

assert ct_arr.shape == lbl_arr.shape, "Shape mismatch between CT and label!"
print("\n✓ Shapes match.")

FileNotFoundError: No such file or no access: '/home/chest_ct/code/models/nnu-net/storage/nnUNet_raw/Dataset104_Nodules/imagesTr/train_1696_a_2.nii.gz'

## 3. Plan and preprocess

This extracts the dataset fingerprint, automatically selects patch sizes / batch sizes / normalisation, and writes preprocessed data to `nnUNet_preprocessed`.

> **Runtime:** expect 10–60 min depending on the number of cases and disk speed.

In [ ]:
# Verify integrity and run plan+preprocess in one step.
# Remove --verify_dataset_integrity after the first successful run.


# Training
CONFIGURATION = "3d_fullres" # or "2d" / "3d_lowres" — see notes below
N_FOLDS       = 5
CUDA_DEVICE   = 0            # GPU index; ignored when device="cpu"

!nnUNetv2_plan_and_preprocess \
    -d {DATASET_ID} \
    --verify_dataset_integrity \
    -c 3d_fullres 2d \
    --no_pbar

In [ ]:
# Inspect what was planned
import json, pathlib

plans_path = pathlib.Path(NNUNET_PREPROCESSED) / DATASET_FULL_NAME / "nnUNetPlans.json"
with open(plans_path) as f:
    plans = json.load(f)

for cfg_name, cfg in plans["configurations"].items():
    print(f"\n── {cfg_name} ─────────────────────")
    print(f"  patch_size          : {cfg.get('patch_size')}")
    print(f"  batch_size          : {cfg.get('batch_size')}")
    print(f"  normalization       : {cfg.get('normalization_schemes')}")
    print(f"  spacing             : {cfg.get('spacing')}")

## 4. Train — 5-fold cross-validation

Each fold takes several hours on a modern GPU. Run folds in parallel across multiple GPUs by opening separate terminals (or Jupyter kernels) and setting `CUDA_VISIBLE_DEVICES` for each.

`--npz` saves validation softmax probabilities — **required** for `nnUNetv2_find_best_configuration`.

In [ ]:
# ── Option A: run all folds sequentially in this notebook ──────────
# Comment this cell out if you prefer to run folds in separate terminals.

for fold in range(N_FOLDS):
    print(f"\n{'='*60}")
    print(f"  Starting fold {fold}  ({CONFIGURATION})")
    print(f"{'='*60}")
    !CUDA_VISIBLE_DEVICES={CUDA_DEVICE} nnUNetv2_train \
        {DATASET_ID} {CONFIGURATION} {fold} \
        --npz

In [ ]:
# ── Option B: copy-paste into separate terminals for parallel training ──
for fold in range(N_FOLDS):
    gpu = fold % 2  # example: 2-GPU machine, round-robin
    cmd = (
        f"CUDA_VISIBLE_DEVICES={gpu} "
        f"nnUNetv2_train {DATASET_ID} {CONFIGURATION} {fold} --npz"
    )
    print(cmd)

In [ ]:
# ── Monitor training progress ──────────────────────────────────────
# Check the latest Dice and loss for each completed fold.

import json, pathlib, re

results_root = pathlib.Path(NNUNET_RESULTS)

for fold in range(N_FOLDS):
    summary_path = (
        results_root
        / DATASET_FULL_NAME
        / f"nnUNetTrainer__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{fold}"
        / "validation"
        / "summary.json"
    )
    if summary_path.exists():
        with open(summary_path) as f:
            s = json.load(f)
        # nnU-Net v2 stores per-class metrics under "metric_per_case" and "foreground_mean"
        fg = s.get("foreground_mean", {})
        dice = fg.get("Dice", fg.get("dice", "N/A"))
        print(f"Fold {fold} | Validation Dice (foreground mean): {dice}")
    else:
        print(f"Fold {fold} | summary.json not found (training not complete)")

## 5. Find the best configuration (optional but recommended)

In [ ]:
# This compares all trained configurations and writes inference_instructions.txt
!nnUNetv2_find_best_configuration {DATASET_ID} --disable_ensembling

In [ ]:
# Read the recommended inference command
import pathlib

instructions_path = pathlib.Path(NNUNET_RESULTS) / DATASET_FULL_NAME / "inference_instructions.txt"
if instructions_path.exists():
    print(instructions_path.read_text())
else:
    print("inference_instructions.txt not found — did find_best_configuration complete?")

## 6. Inference

Input images must follow the same naming convention used during training:
`<case_id>_0000.nii.gz`

In [ ]:
# ── Define inference I/O ───────────────────────────────────────────
INFER_INPUT_FOLDER  = str(images_ts)   # imagesTs prepared during conversion
INFER_OUTPUT_FOLDER = os.path.join(NNUNET_RESULTS, DATASET_FULL_NAME, "predictions")

os.makedirs(INFER_OUTPUT_FOLDER, exist_ok=True)
print("Input :", INFER_INPUT_FOLDER)
print("Output:", INFER_OUTPUT_FOLDER)

In [ ]:
# ── Run inference using the 5-fold ensemble ────────────────────────
!nnUNetv2_predict \
    -i  "{INFER_INPUT_FOLDER}" \
    -o  "{INFER_OUTPUT_FOLDER}" \
    -d  {DATASET_ID} \
    -c  {CONFIGURATION} \
    --save_probabilities

In [ ]:
# ── Apply postprocessing (if find_best_configuration was run) ──────
import glob, pathlib

trainer_folder = (
    pathlib.Path(NNUNET_RESULTS)
    / DATASET_FULL_NAME
    / f"nnUNetTrainer__nnUNetPlans__{CONFIGURATION}"
)

pp_pkl = list(trainer_folder.glob("**/postprocessing.pkl"))
if pp_pkl:
    pp_pkl = pp_pkl[0]
    plans_json   = trainer_folder / "plans.json"
    dataset_json = pathlib.Path(DATASET_RAW_DIR) / "dataset.json"

    INFER_PP_OUTPUT = INFER_OUTPUT_FOLDER + "_pp"
    os.makedirs(INFER_PP_OUTPUT, exist_ok=True)

    !nnUNetv2_apply_postprocessing \
        -i  "{INFER_OUTPUT_FOLDER}" \
        -o  "{INFER_PP_OUTPUT}" \
        --pp_pkl_file "{pp_pkl}" \
        -plans_json "{plans_json}" \
        -dataset_json "{dataset_json}"
else:
    print("No postprocessing.pkl found — skipping. Run find_best_configuration first.")

## 7. Evaluate predictions (if ground-truth labels exist for test set)

In [ ]:
# ── Convert test masks to binary labels first ──────────────────────
# Skip this cell if your test split has no ground-truth labels.

TEST_LABELS_DIR = pathlib.Path(DATASET_RAW_DIR) / "labelsTs"
TEST_LABELS_DIR.mkdir(exist_ok=True)

for entry in tqdm(split_data.get("test", []), desc="Test labels"):
    case_id  = entry["name"].replace(".nii.gz", "")
    seg_path = entry["seg_path"]
    ggo_chs  = entry["ggo_channels"]
    lbl_dest = TEST_LABELS_DIR / f"{case_id}.nii.gz"

    if not lbl_dest.exists() and os.path.isfile(seg_path):
        binary, affine, header = extract_ggo_mask(seg_path, ggo_chs)
        lbl_img = nib.Nifti1Image(binary, affine, header)
        lbl_img.set_data_dtype(np.uint8)
        nib.save(lbl_img, str(lbl_dest))

print("Test labels written.")

In [ ]:
# ── Compute Dice / IoU on the test set ────────────────────────────
import numpy as np
import nibabel as nib
from pathlib import Path


def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    pred = (pred > 0).astype(np.uint8)
    gt   = (gt   > 0).astype(np.uint8)
    intersection = (pred & gt).sum()
    denom = pred.sum() + gt.sum()
    return (2.0 * intersection / denom) if denom > 0 else 1.0


def iou_score(pred: np.ndarray, gt: np.ndarray) -> float:
    pred = (pred > 0).astype(np.uint8)
    gt   = (gt   > 0).astype(np.uint8)
    intersection = (pred & gt).sum()
    union = (pred | gt).sum()
    return (intersection / union) if union > 0 else 1.0


pred_dir  = Path(INFER_PP_OUTPUT if pp_pkl else INFER_OUTPUT_FOLDER)
label_dir = TEST_LABELS_DIR

dice_scores, iou_scores = [], []
missing = []

for lbl_path in sorted(label_dir.glob("*.nii.gz")):
    case_id   = lbl_path.stem.replace(".nii", "")
    pred_path = pred_dir / lbl_path.name

    if not pred_path.exists():
        missing.append(case_id)
        continue

    gt_arr   = nib.load(str(lbl_path)).get_fdata()
    pred_arr = nib.load(str(pred_path)).get_fdata()

    d = dice_score(pred_arr, gt_arr)
    i = iou_score(pred_arr, gt_arr)
    dice_scores.append(d)
    iou_scores.append(i)

print(f"Evaluated {len(dice_scores)} cases")
if missing:
    print(f"Missing predictions for: {missing}")
if dice_scores:
    print(f"\nDice  — mean: {np.mean(dice_scores):.4f}  std: {np.std(dice_scores):.4f}  "
          f"min: {np.min(dice_scores):.4f}  max: {np.max(dice_scores):.4f}")
    print(f"IoU   — mean: {np.mean(iou_scores):.4f}  std: {np.std(iou_scores):.4f}  "
          f"min: {np.min(iou_scores):.4f}  max: {np.max(iou_scores):.4f}")

## 8. Visualise a prediction

In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from pathlib import Path


def show_case(
    ct_path: str,
    pred_path: str,
    gt_path: str | None = None,
    n_slices: int = 6,
    hu_window: tuple[float, float] = (-1000, 400),
):
    ct   = nib.load(ct_path).get_fdata()
    pred = nib.load(pred_path).get_fdata()
    gt   = nib.load(gt_path).get_fdata() if gt_path else None

    # Find slices that contain GGO foreground
    ggo_slices = np.where(pred.sum(axis=(0, 1)) > 0)[0]
    if len(ggo_slices) == 0:
        ggo_slices = np.linspace(0, ct.shape[2]-1, n_slices, dtype=int)
    else:
        idx = np.linspace(0, len(ggo_slices)-1, n_slices, dtype=int)
        ggo_slices = ggo_slices[idx]

    ncols = n_slices
    nrows = 3 if gt is not None else 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    fig.suptitle(Path(ct_path).name, fontsize=12)

    hu_min, hu_max = hu_window

    for col, sl in enumerate(ggo_slices):
        ct_sl   = np.clip(ct[:, :, sl], hu_min, hu_max)
        ct_norm = (ct_sl - hu_min) / (hu_max - hu_min)

        axes[0, col].imshow(ct_norm.T, cmap="gray", origin="lower")
        axes[0, col].set_title(f"CT  z={sl}", fontsize=8)
        axes[0, col].axis("off")

        axes[1, col].imshow(ct_norm.T, cmap="gray", origin="lower")
        axes[1, col].imshow(pred[:, :, sl].T, cmap="Reds", alpha=0.45, origin="lower", vmin=0, vmax=1)
        axes[1, col].set_title("Prediction", fontsize=8)
        axes[1, col].axis("off")

        if gt is not None:
            axes[2, col].imshow(ct_norm.T, cmap="gray", origin="lower")
            axes[2, col].imshow(gt[:, :, sl].T, cmap="Greens", alpha=0.45, origin="lower", vmin=0, vmax=1)
            axes[2, col].set_title("Ground truth", fontsize=8)
            axes[2, col].axis("off")

    plt.tight_layout()
    plt.show()


# ── Pick the first test case that has a prediction ─────────────────
pred_dir = Path(INFER_PP_OUTPUT if pp_pkl else INFER_OUTPUT_FOLDER)
preds    = sorted(pred_dir.glob("*.nii.gz"))

if preds:
    first_case = preds[0].stem.replace(".nii", "")
    ct_file    = images_ts / f"{first_case}_0000.nii.gz"
    gt_file    = TEST_LABELS_DIR / f"{first_case}.nii.gz"

    show_case(
        ct_path   = str(ct_file),
        pred_path = str(preds[0]),
        gt_path   = str(gt_file) if gt_file.exists() else None,
    )
else:
    print("No predictions found.")

## 9. Export and portability

In [ ]:
# Pack the trained model into a zip for transfer or sharing
MODEL_ZIP = os.path.join(NNUNET_RESULTS, f"{DATASET_FULL_NAME}_model.zip")

!nnUNetv2_export_model_to_zip \
    -d {DATASET_ID} \
    -o "{MODEL_ZIP}"

print("Model exported to:", MODEL_ZIP)

In [ ]:
# On the target machine, install with:
# !nnUNetv2_install_pretrained_model_from_zip "{MODEL_ZIP}"
print(f"To install on another machine:\n"
      f"  nnUNetv2_install_pretrained_model_from_zip {MODEL_ZIP}")

---
## Appendix: Common issues

| Issue | Fix |
|---|---|
| `nnUNet_raw not set` | Re-run cell 1 (sets `os.environ` for this kernel) |
| `dataset.json` errors | Re-run the `generate_dataset_json` cell |
| CUDA OOM during training | Reduce `nnUNet_n_proc_DA` or use `--c 3d_lowres` |
| NaN loss | Check CT values — extreme HU outliers can destabilise normalisation; clip to [-1024, 3071] before conversion |
| Very low Dice on GGO | GGO is subtle and diffuse; try ensembling 3d_fullres + 2d, and consider a lower `foreground_threshold` in postprocessing |
| Mask shape mismatch | Confirm the segmentation is `F×H×W×D` not transposed — adjust `extract_ggo_mask` indexing if needed |
| Variable voxel spacing | nnU-Net handles this automatically via anisotropic resampling; check spacing in the fingerprint |
